# 第 4 章｜Resource 查詢

依序執行每一格；可修改標示的參數後重跑。

## 執行前：可替換設定總覽

以下項目都可以依測試環境替換：

- `.env` Provider：正式課程填 `CILLM_API_KEY`；只有個人本機測試才填 `OPENAI_API_KEY`；兩者都有時 CILLM 優先。
- CILLM：`CILLM_BASE_URL`、`CILLM_USER_ID`、`CILLM_PLATFORM`、`CILLM_AGENT`、`GPT_OSS_MODEL_NAME`、`GEMMA_MODEL_NAME`、`CILLM_VISION_FORMAT`。
- CILLM 圖片：正式課程使用 `CILLM_VISION_FORMAT=blocks`，以標準 `image_url` content blocks 交給 Gemma 4 31B IT。
- OpenAI：`OPENAI_API_KEY`、`OPENAI_MODEL_NAME`；預設 `gpt-4o`，僅供個人筆電模擬測試。
- 密碼式 AES：第 6 章由使用者在 Notebook 隱藏輸入設定保險庫密碼；密碼不寫入 `.env` 或加密檔。
- 路徑：只有從其他工作目錄啟動 Notebook 時才需要調整 `ROOT`；一般從教材根目錄或 `notebooks/` 啟動不必修改。

> 請勿把含有真實 Key 的 `.env`、Notebook 輸出或截圖提交到 Git。

### 本章可替換

- `USER_REQUEST`：替換資源查詢問題。
- Resource 內容：替換 `resources/` 下的文字檔。
- `RESOURCE_CATALOG`：可在 `course_utils.py` 替換資源路徑、名稱與路由關鍵字。
- `ask_gpt_oss` 的 instruction：可替換回答語言、格式與引用限制。

In [8]:
from pathlib import Path
import importlib, os, sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import course_utils
importlib.reload(course_utils)
from course_utils import *
print("教材根目錄：", ROOT)
connection = verify_cillm_key()
print("✅ API Key 驗證成功")
print("目前 Provider：", connection["provider"])
print("目前模型：", connection["model"])
print("首次回覆：", connection["reply"])

教材根目錄： C:\Users\rathe\Project\cillm\CILLM_Workshop\Lecture03
✅ API Key 驗證成功
目前 Provider： openai
目前模型： gpt-4o
首次回覆： CILLM 連線成功


## 檢查目前 API Key 的基本 Scope

下方 Cell 會查詢並列出這支 Key 實際具備的 RBAC scopes。本教材呼叫 GPT-OSS 至少需要 `llm.chat`。

In [9]:
scopes = get_current_key_scopes()
if scopes is None:
    print("目前使用 OpenAI API；CILLM RBAC scope 不適用。")
else:
    print("目前 CILLM_API_KEY 具備的 scopes：")
    for scope in scopes:
        print("-", scope)
    required_scope = "llm.chat"
    print(f"✅ 已具備教材基本 scope：{required_scope}" if "*" in scopes or required_scope in scopes else f"❌ 缺少教材基本 scope：{required_scope}")

目前使用 OpenAI API；CILLM RBAC scope 不適用。


> **執行模式 Hint**
>
> - 本教材每章第一格會取得 `CILLM_API_KEY`；若 `.env` 未設定，Notebook 會以隱藏輸入提示使用者填入。
> - 取得 Key 後會立即呼叫一次 `openai/gpt-oss-120b`，確認 Key 與連線可用。
> - `CILLM_BASE_URL` 預設沿用 Lecture 02 的測試端點；缺少或無效 Key 時會立即停止。
> - 正式課程的 CILLM 模式：文字使用 GPT-OSS，圖片使用 NVIDIA NIM 的 `google/gemma-4-31b-it`，不會 fallback 到 OpenAI。
> - `CILLM_VISION_FORMAT=blocks` 會送出 NVIDIA Gemma 4 NIM 使用的標準 OpenAI `image_url` content blocks。
> - CILLM gateway 必須部署可接受陣列 content 的新版 schema；`html` 僅保留為特殊相容模式，不是正式課程預設。
> - 只有 OpenAI Key 時，才以 `gpt-4o` 模擬文字與圖片流程，供個人筆電測試。

> **Resource Hint**
>
> - 問延誤、餐點或行李：預期選到旅客服務規範。
> - 問雷雨、飛航計畫：預期選到航班作業規範。
> - 問 API key、密碼：預期選到資訊安全規範。
> - 問請假或設備借用：預期選到員工行政規範。

## 兩階段 Resource Access：Router 選文件，Retriever 找段落

- **Resource Router**：AI 根據 descriptions 選擇要開哪一份 Resource，並由 Pydantic 驗證。
- **Retriever / `search_resource`**：只在已選文件內挑出相關條文。
- **Answer LLM**：只使用 Retriever 提供的條文組織回答。

問題一次只啟用一題；請註解目前 `USER_REQUEST`，再取消另一題的註解。

In [14]:
# 預設：旅客延誤服務
# USER_REQUEST = "航班延誤多久要提供餐飲券？"

# 其他 Resource 情境：每次只取消其中一行註解
# USER_REQUEST = "特殊餐點需要多久以前申請？"
USER_REQUEST = "雷雨警報時，航務人員要重新評估什麼？"
# USER_REQUEST = "維修工具短少時可以放行航機嗎？"
# USER_REQUEST = "API key 應該存在哪裡？如果外洩該怎麼辦？"
# USER_REQUEST = "病假在什麼情況下需要補交證明？"
# USER_REQUEST = "公司的客服時間和緊急航班資訊來源是什麼？"

# Stage 1: Router 選擇 Resource
resource_route = route_resource(USER_REQUEST)
print("Stage 1 - Pydantic 驗證後的 Resource Router 結果：")
show(resource_route.model_dump())
if resource_route.resource_name == "none":
    raise RuntimeError("AI 判斷不需要 Resource，無法繼續本範例。")

# Stage 2: Retriever 從已選文件中找相關條文
resource_name = resource_route.resource_name
context = search_resource(resource_name, USER_REQUEST)
print("\nStage 2 - Retriever 載入的 Resource：", resource_name)
print("Stage 2 - 命中的相關條文：\n", context)

# Stage 3: LLM 只依據命中條文回答
answer = ask_gpt_oss(USER_REQUEST, context, "只能根據提供的 Resource 條文回答；若沒有答案就明確說明。使用繁體中文。")
print("\nStage 3 - LLM 最終回答：\n", answer)

Stage 1 - Pydantic 驗證後的 Resource Router 結果：
{
  "resource_name": "flight_operations_guide",
  "reason": "使用者問題涉及雷雨警報和航務人員的操作，這與航班、航務、起飛、飛航計畫、雷雨與異常通報作業相關，因此選擇 flight_operations_guide 資源。"
}

Stage 2 - Retriever 載入的 Resource： flight_operations_guide
Stage 2 - 命中的相關條文：
 3. 雷雨警報時須重新評估航路。

Stage 3 - LLM 最終回答：
 雷雨警報時，航務人員要重新評估航路。


In [12]:
print_execution_trace(question=USER_REQUEST, tool="✗ 本章停用", resource=f"✓ {resource_name} (Pydantic routed)", answer=answer)

AI Agent 執行追蹤
使用者問題：
航班延誤多久要提供餐飲券？

是否需要工具：
✗ 本章停用

是否需要資源：
✓ passenger_service_rules (Pydantic routed)

最終回答：
根據提供的資源，航班延誤達 120 分鐘時需提供餐飲券。

